In [0]:
# Setting the raw data location and defining the Databricks tables where cleaned outputs are being saved

from pyspark.sql import functions as F

RAW_BASE_PATH = "/Volumes/mortgage_risk/data_files/raw/historical_data_2024"

spark.sql("CREATE SCHEMA IF NOT EXISTS mortgage_risk.analytics")

ORIGINATION_TABLE = "mortgage_risk.analytics.origination_2024"
PERFORMANCE_TABLE = "mortgage_risk.analytics.performance_2024"
ANALYSIS_BASE_TABLE = "mortgage_risk.analytics.analysis_base_2024"

In [0]:
# Defining the source column layouts so raw text files are being read with the correct structure
ORIGINATION_COLUMNS = ["credit_score", "first_payment_date", "first_time_homebuyer_flag", 
                       "maturity_date", "msa", "mortgage_insurance_percentage","number_of_units", "occupancy_status", "combined_loan_to_value","debt_to_income_ratio", "original_upb", "loan_to_value","original_interest_rate", "channel", "prepayment_penalty_mortgage_flag", "product_type", "property_state","property_type", "postal_code", "loan_sequence_number", "loan_purpose","original_loan_term", "number_of_borrowers", "seller_name", "servicer_name", "super_conforming_flag", "pre_harp_loan_sequence_number", "program_indicator", "harp_indicator","property_valuation_method", "interest_only_indicator","mortgage_insurance_cancellation_indicator"
]

PERFORMANCE_COLUMNS = ["loan_sequence_number", "monthly_reporting_period", 
                       "current_actual_upb", "current_loan_delinquency_status", "loan_age","remaining_months_to_maturity", "repurchase_flag", "modification_flag", "zero_balance_code", "zero_balance_effective_date", "current_interest_rate", "current_deferred_upb", "due_date_of_last_paid_installment", "mi_recoveries", "net_sale_proceeds", "non_mi_recoveries", "expenses", "legal_costs", "maintenance_and_preservation_costs", "taxes_and_insurance", "miscellaneous_expenses","actual_loss_calculation", "modification_cost", "step_modification_flag", "deferred_payment_plan", "estimated_loan_to_value", "zero_balance_removal_upb", "delinquent_accrued_interest", "delinquency_due_to_disaster","borrower_assistance_status_code", "current_month_modification_cost","interest_bearing_upb",
]


In [0]:
# Reading raw origination and performance files from the volume and tagging each row with its source quarter
origination_raw_df = (spark.read
                            .option("sep", "|")
                            .option("header", "false")
                            .csv(f"{RAW_BASE_PATH}/*/historical_data_2024Q*.txt")
                            .toDF(*ORIGINATION_COLUMNS)
                            .withColumn("source_file", F.col("_metadata.file_path"))
                            .withColumn("quarter", F.regexp_extract("source_file", r"(2024Q[1-4])", 1))
)

performance_raw_df = (spark.read
                            .option("sep", "|")
                            .option("header", "false")
                            .csv(f"{RAW_BASE_PATH}/*/historical_data_time_2024Q*.txt")
                            .toDF(*PERFORMANCE_COLUMNS)
                            .withColumn("source_file", F.col("_metadata.file_path"))
                            .withColumn("quarter", F.regexp_extract("source_file", r"(2024Q[1-4])", 1))
)

In [0]:
# Selecting the origination fields needed for analysis and converting them into usable numeric and date types
origination_df = (
    origination_raw_df.select("loan_sequence_number", "quarter", "credit_score", "first_payment_date", 
                              "first_time_homebuyer_flag", "msa","occupancy_status", "combined_loan_to_value", "debt_to_income_ratio", "original_upb", "loan_to_value","original_interest_rate", "channel", "product_type","property_state", "property_type", "loan_purpose","original_loan_term", "number_of_borrowers","super_conforming_flag","interest_only_indicator"
    ).withColumn("credit_score", F.col("credit_score").cast("int"))
     .withColumn("combined_loan_to_value", F.col("combined_loan_to_value").cast("double"))
     .withColumn("debt_to_income_ratio", F.col("debt_to_income_ratio").cast("double"))
     .withColumn("original_upb", F.col("original_upb").cast("double"))
     .withColumn("loan_to_value", F.col("loan_to_value").cast("double"))
     .withColumn("original_interest_rate", F.col("original_interest_rate").cast("double"))
     .withColumn("original_loan_term", F.col("original_loan_term").cast("int"))
     .withColumn("number_of_borrowers", F.col("number_of_borrowers").cast("int"))
     .withColumn("first_payment_date", F.to_date(F.col("first_payment_date"), "yyyyMM"))
)

In [0]:
# Selecting the performance fields needed for analysis and standardizing their numeric and date formats

performance_df = (
    performance_raw_df.select("loan_sequence_number", "quarter", "monthly_reporting_period", 
                              "current_actual_upb", "current_loan_delinquency_status","loan_age", "remaining_months_to_maturity", "zero_balance_code", "zero_balance_effective_date", "current_interest_rate", "modification_flag", "estimated_loan_to_value", "delinquency_due_to_disaster","borrower_assistance_status_code", "interest_bearing_upb"
    ).withColumn("monthly_reporting_period", F.to_date(F.col("monthly_reporting_period"), "yyyyMM"))
     .withColumn("zero_balance_effective_date", F.to_date(F.col("zero_balance_effective_date"), "yyyyMM"))
     .withColumn("current_actual_upb", F.col("current_actual_upb").cast("double"))
     .withColumn("loan_age", F.col("loan_age").cast("int"))
     .withColumn("remaining_months_to_maturity", F.col("remaining_months_to_maturity").cast("int"))
     .withColumn("current_interest_rate", F.col("current_interest_rate").cast("double"))
     .withColumn("estimated_loan_to_value", F.col("estimated_loan_to_value").cast("double"))
     .withColumn("interest_bearing_upb", F.col("interest_bearing_upb").cast("double"))
)

In [0]:
# Joining monthly performance history with origination attributes to create one analysis-ready base table
analysis_base_df = (
    performance_df.join(origination_df, on = "loan_sequence_number", how = "inner")
                  .select("loan_sequence_number", 
                          performance_df.quarter.alias("performance_quarter"), 
                          "monthly_reporting_period", "current_actual_upb", "current_loan_delinquency_status", "loan_age", "remaining_months_to_maturity", "zero_balance_code", "zero_balance_effective_date", "current_interest_rate", "modification_flag", "estimated_loan_to_value", "delinquency_due_to_disaster", "borrower_assistance_status_code", "interest_bearing_upb", 
                          origination_df.quarter.alias("origination_quarter"), 
                          "first_payment_date", "credit_score", "first_time_homebuyer_flag", "msa", "occupancy_status", "combined_loan_to_value", "debt_to_income_ratio", "original_upb", "loan_to_value", "original_interest_rate", "channel", "product_type", "property_state", "property_type", "loan_purpose", "original_loan_term", "number_of_borrowers", "super_conforming_flag", "interest_only_indicator"
    )
)

In [0]:
# Checking row counts and distinct loan counts to confirm the ingestion and join are behaving as expected
print("Origination row count :", origination_df.count())
print("Performance row count :", performance_df.count())
print("Analysis base row count :", analysis_base_df.count())

print("Distinct origination loans :", origination_df.select("loan_sequence_number").distinct().count())
print("Distinct performance loans :", performance_df.select("loan_sequence_number").distinct().count())
print("Distinct joined loans :", analysis_base_df.select("loan_sequence_number").distinct().count())

Origination row count : 1048407
Performance row count : 14226973
Analysis base row count : 14226973
Distinct origination loans : 1048407
Distinct performance loans : 1048407
Distinct joined loans : 1048407


In [0]:
display(origination_df.limit(5))
display(performance_df.limit(5))
display(analysis_base_df.limit(5))

loan_sequence_number,quarter,credit_score,first_payment_date,first_time_homebuyer_flag,msa,occupancy_status,combined_loan_to_value,debt_to_income_ratio,original_upb,loan_to_value,original_interest_rate,channel,product_type,property_state,property_type,loan_purpose,original_loan_term,number_of_borrowers,super_conforming_flag,interest_only_indicator
F24Q10000001,2024Q1,747,2024-03-01,N,null,P,67.0,26.0,82000.0,67.0,8.0,R,FRM,KS,MH,P,360,1,null,N
F24Q10000002,2024Q1,710,2024-05-01,N,null,P,55.0,36.0,295000.0,55.0,7.125,R,FRM,IL,SF,C,360,1,null,N
F24Q10000003,2024Q1,659,2024-03-01,N,null,P,68.0,32.0,305000.0,68.0,7.875,R,FRM,MI,SF,P,360,2,null,N
F24Q10000004,2024Q1,784,2024-03-01,N,46540,P,76.0,50.0,125000.0,76.0,6.75,R,FRM,NY,SF,C,360,1,null,N
F24Q10000005,2024Q1,775,2024-03-01,N,46540,I,70.0,15.0,154000.0,70.0,7.625,R,FRM,NY,SF,C,360,2,null,N


loan_sequence_number,quarter,monthly_reporting_period,current_actual_upb,current_loan_delinquency_status,loan_age,remaining_months_to_maturity,zero_balance_code,zero_balance_effective_date,current_interest_rate,modification_flag,estimated_loan_to_value,delinquency_due_to_disaster,borrower_assistance_status_code,interest_bearing_upb
F24Q10000001,2024Q1,2024-02-01,82000.0,0,0,360,null,null,8.0,null,999.0,null,null,82000.0
F24Q10000001,2024Q1,2024-03-01,82000.0,0,1,359,null,null,8.0,null,999.0,null,null,82000.0
F24Q10000001,2024Q1,2024-04-01,82000.0,0,2,358,null,null,8.0,null,999.0,null,null,82000.0
F24Q10000001,2024Q1,2024-05-01,82000.0,0,3,357,null,null,8.0,null,999.0,null,null,82000.0
F24Q10000001,2024Q1,2024-06-01,82000.0,0,4,356,null,null,8.0,null,999.0,null,null,82000.0


loan_sequence_number,performance_quarter,monthly_reporting_period,current_actual_upb,current_loan_delinquency_status,loan_age,remaining_months_to_maturity,zero_balance_code,zero_balance_effective_date,current_interest_rate,modification_flag,estimated_loan_to_value,delinquency_due_to_disaster,borrower_assistance_status_code,interest_bearing_upb,origination_quarter,first_payment_date,credit_score,first_time_homebuyer_flag,msa,occupancy_status,combined_loan_to_value,debt_to_income_ratio,original_upb,loan_to_value,original_interest_rate,channel,product_type,property_state,property_type,loan_purpose,original_loan_term,number_of_borrowers,super_conforming_flag,interest_only_indicator
F24Q10000006,2024Q1,2024-02-01,51000.0,0,0,360,null,null,7.25,null,87.0,null,null,51000.0,2024Q1,2024-03-01,815,Y,null,P,88.0,49.0,51000.0,88.0,7.25,R,FRM,IL,SF,P,360,1,null,N
F24Q10000006,2024Q1,2024-03-01,51000.0,0,1,359,null,null,7.25,null,87.0,null,null,51000.0,2024Q1,2024-03-01,815,Y,null,P,88.0,49.0,51000.0,88.0,7.25,R,FRM,IL,SF,P,360,1,null,N
F24Q10000006,2024Q1,2024-04-01,51000.0,0,2,358,null,null,7.25,null,84.0,null,null,51000.0,2024Q1,2024-03-01,815,Y,null,P,88.0,49.0,51000.0,88.0,7.25,R,FRM,IL,SF,P,360,1,null,N
F24Q10000006,2024Q1,2024-05-01,50000.0,0,3,357,null,null,7.25,null,79.0,null,null,50000.0,2024Q1,2024-03-01,815,Y,null,P,88.0,49.0,51000.0,88.0,7.25,R,FRM,IL,SF,P,360,1,null,N
F24Q10000006,2024Q1,2024-06-01,50000.0,0,4,356,null,null,7.25,null,78.0,null,null,50000.0,2024Q1,2024-03-01,815,Y,null,P,88.0,49.0,51000.0,88.0,7.25,R,FRM,IL,SF,P,360,1,null,N


In [0]:
# Saving the cleaned and curated datasets as managed Delta tables in Databricks
# - Partitioning by quarter so the tables are being stored more efficiently for later reads
# - Overwriting the tables so the latest ingestion run is replacing older versions cleanly

(
    origination_df.write
                  .format("delta")
                  .mode("overwrite")
                  .partitionBy("quarter")
                  .saveAsTable(ORIGINATION_TABLE)
)

(
    performance_df.write
                  .format("delta")
                  .mode("overwrite")
                  .partitionBy("quarter")
                  .saveAsTable(PERFORMANCE_TABLE)
)

(
    analysis_base_df.write
                    .format("delta")
                    .mode("overwrite")
                    .partitionBy("performance_quarter")
                    .saveAsTable(ANALYSIS_BASE_TABLE)
)

In [0]:
print("Delta Tables written successfully.")

display(spark.table(ORIGINATION_TABLE).limit(5))
display(spark.table(PERFORMANCE_TABLE).limit(5))
display(spark.table(ANALYSIS_BASE_TABLE).limit(5))

Delta Tables written successfully.


loan_sequence_number,quarter,credit_score,first_payment_date,first_time_homebuyer_flag,msa,occupancy_status,combined_loan_to_value,debt_to_income_ratio,original_upb,loan_to_value,original_interest_rate,channel,product_type,property_state,property_type,loan_purpose,original_loan_term,number_of_borrowers,super_conforming_flag,interest_only_indicator
F24Q20000001,2024Q2,701,2024-06-01,N,null,P,36.0,48.0,512000.0,36.0,6.0,R,FRM,CO,SF,N,360,2,null,N
F24Q20000002,2024Q2,804,2024-06-01,N,30460,P,75.0,34.0,244000.0,75.0,6.5,R,FRM,KY,SF,P,360,1,null,N
F24Q20000003,2024Q2,754,2024-06-01,Y,37620,P,95.0,44.0,149000.0,95.0,6.25,B,FRM,WV,MH,P,360,1,null,N
F24Q20000004,2024Q2,779,2024-06-01,N,30340,P,27.0,39.0,100000.0,27.0,6.375,R,FRM,ME,SF,P,360,2,null,N
F24Q20000005,2024Q2,730,2024-07-01,N,null,P,73.0,41.0,540000.0,73.0,6.75,R,FRM,CO,SF,N,360,2,null,N


loan_sequence_number,quarter,monthly_reporting_period,current_actual_upb,current_loan_delinquency_status,loan_age,remaining_months_to_maturity,zero_balance_code,zero_balance_effective_date,current_interest_rate,modification_flag,estimated_loan_to_value,delinquency_due_to_disaster,borrower_assistance_status_code,interest_bearing_upb
F24Q20000001,2024Q2,2024-05-01,512000.0,0,0,360,null,null,6.0,null,35.0,null,null,512000.0
F24Q20000001,2024Q2,2024-06-01,512000.0,0,1,359,null,null,6.0,null,35.0,null,null,512000.0
F24Q20000001,2024Q2,2024-07-01,511000.0,0,2,358,null,null,6.0,null,36.0,null,null,511000.0
F24Q20000001,2024Q2,2024-08-01,511000.0,0,3,357,null,null,6.0,null,36.0,null,null,511000.0
F24Q20000001,2024Q2,2024-09-01,510000.0,0,4,356,null,null,6.0,null,36.0,null,null,510000.0


loan_sequence_number,performance_quarter,monthly_reporting_period,current_actual_upb,current_loan_delinquency_status,loan_age,remaining_months_to_maturity,zero_balance_code,zero_balance_effective_date,current_interest_rate,modification_flag,estimated_loan_to_value,delinquency_due_to_disaster,borrower_assistance_status_code,interest_bearing_upb,origination_quarter,first_payment_date,credit_score,first_time_homebuyer_flag,msa,occupancy_status,combined_loan_to_value,debt_to_income_ratio,original_upb,loan_to_value,original_interest_rate,channel,product_type,property_state,property_type,loan_purpose,original_loan_term,number_of_borrowers,super_conforming_flag,interest_only_indicator
F24Q20000003,2024Q2,2024-05-01,149000.0,0,0,360,null,null,6.25,null,999.0,null,null,149000.0,2024Q2,2024-06-01,754,Y,37620,P,95.0,44.0,149000.0,95.0,6.25,B,FRM,WV,MH,P,360,1,null,N
F24Q20000003,2024Q2,2024-06-01,149000.0,0,1,359,null,null,6.25,null,999.0,null,null,149000.0,2024Q2,2024-06-01,754,Y,37620,P,95.0,44.0,149000.0,95.0,6.25,B,FRM,WV,MH,P,360,1,null,N
F24Q20000003,2024Q2,2024-07-01,149000.0,0,2,358,null,null,6.25,null,999.0,null,null,149000.0,2024Q2,2024-06-01,754,Y,37620,P,95.0,44.0,149000.0,95.0,6.25,B,FRM,WV,MH,P,360,1,null,N
F24Q20000003,2024Q2,2024-08-01,149000.0,0,3,357,null,null,6.25,null,999.0,null,null,149000.0,2024Q2,2024-06-01,754,Y,37620,P,95.0,44.0,149000.0,95.0,6.25,B,FRM,WV,MH,P,360,1,null,N
F24Q20000003,2024Q2,2024-09-01,148000.0,0,4,356,null,null,6.25,null,999.0,null,null,148000.0,2024Q2,2024-06-01,754,Y,37620,P,95.0,44.0,149000.0,95.0,6.25,B,FRM,WV,MH,P,360,1,null,N
